# Оптимизация гиперпараметров

> Практические задания по оптимизации гиперпараметров с использованием Optuna, Hyperopt, Ray Tune и других библиотек

# 1. Оптимизация гиперпараметров для классических моделей машинного обучения

## Задание 1: Оптимизация Random Forest с Optuna

Описание:
Вы работаете над задачей предсказания цен на недвижимость и хотите оптимизировать гиперпараметры модели RandomForestRegressor.
Требования:

Используйте датасет California Housing Prices.
Определите пространство гиперпараметров для оптимизации:

n_estimators: [50, 500]
max_depth: [3, 20]
min_samples_split: [2, 20]
min_samples_leaf: [1, 10]

Используйте Optuna для поиска оптимальных гиперпараметров:

```
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10)
    }
    model = RandomForestRegressor(**params)
    score = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)
```

Визуализируйте результаты оптимизации:

optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)

Обучите финальную модель с лучшими гиперпараметрами и оцените её качество на тестовой выборке.
Дополнительно:

Добавьте раннюю остановку (pruning) с помощью optuna.pruners.MedianPruner.

https://www.kaggle.com/datasets/camnugent/california-housing-prices

## Задание 2: Оптимизация XGBoost с Hyperopt

Описание:
Вы работаете над задачей классификации отзывов и хотите оптимизировать XGBoost.
Требования:

Используйте датасет IMDB Reviews.
Преобразуйте текстовые данные в числовые признаки с помощью TF-IDF.
Определите пространство гиперпараметров:

learning_rate: [0.01, 0.3]
max_depth: [3, 10]
n_estimators: [50, 500]
subsample: [0.6, 1.0]
colsample_bytree: [0.6, 1.0]

Используйте Hyperopt для оптимизации:

```
from hyperopt import fmin, tpe, hp, Trials
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

space = {
    'learning_rate': hp.loguniform('learning_rate', -2, -0.5),
    'max_depth': hp.quniform('max_depth', 3, 10, 1),
    'n_estimators': hp.quniform('n_estimators', 50, 500, 50),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0)
}

def objective(params):
    model = XGBClassifier(**params)
    score = cross_val_score(model, X, y, cv=5, scoring='accuracy').mean()
    return -score

trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=100, trials=trials)
```

Визуализируйте результаты и обучите финальную модель.
Дополнительно:

Сравните результаты с оптимизацией через Optuna.

https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews


## Задание 3: Оптимизация логистической регрессии с Grid Search и Random Search
Описание:
Вы хотите сравнить результаты оптимизации логистической регрессии с помощью GridSearchCV и RandomizedSearchCV.
Требования:

Используйте датасет Titanic.
Определите пространство гиперпараметров:

C: [0.001, 1, 10, 100]
penalty: ['l1', 'l2', 'elasticnet']
solver: ['liblinear', 'saga']

Примените GridSearchCV и RandomizedSearchCV:

```
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

param_grid = {
    'C': [0.001, 1, 10, 100],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['liblinear', 'saga']
}

grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)

random_search = RandomizedSearchCV(LogisticRegression(), param_grid, cv=5, scoring='accuracy', n_iter=20)
random_search.fit(X, y)
```

Сравните время выполнения и качество моделей.
Дополнительно:

Добавьте в пространство поиска параметр l1_ratio для elasticnet.

https://www.kaggle.com/c/titanic/data


# 2. Оптимизация гиперпараметров для нейронных сетей

## Задание 4: Оптимизация архитектуры нейронной сети с Optuna

Описание:
Вы разрабатываете нейронную сеть для классификации изображений и хотите оптимизировать её архитектуру.
Требования:

Используйте датасет MNIST.
Определите пространство гиперпараметров:

Количество слоёв: [1, 4]
Количество нейронов в слое: [32, 512]
Функция активации: ['relu', 'tanh', 'sigmoid']
learning_rate: [0.0001, 0.1]
batch_size: [32, 256]

Используйте Optuna для оптимизации:

```
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 4)
    n_units = trial.suggest_int('n_units', 32, 512)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh', 'sigmoid'])
    lr = trial.suggest_loguniform('lr', 1e-4, 1e-1)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])

    model = build_model(n_layers, n_units, activation)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(X_train, y_train, batch_size=batch_size)

    for epoch in range(10):
        train_model(model, optimizer, train_loader)

    val_loss = evaluate_model(model, X_val, y_val)
    return val_loss

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
```

Визуализируйте результаты и обучите финальную модель.
Дополнительно:

Добавьте dropout и оптимизируйте его вероятность.

https://www.kaggle.com/datasets/hojjatk/mnist-dataset


## Задание 5: Оптимизация сверточной нейронной сети с Ray Tune

Описание:
Вы хотите оптимизировать архитектуру сверточной нейронной сети для классификации изображений.
Требования:

Используйте датасет CIFAR-10.
Определите пространство гиперпараметров:

Количество свёрточных слоёв: [1, 3]
Количество фильтров: [16, 128]
Размер ядра: [3, 5]
learning_rate: [0.0001, 0.01]

Используйте Ray Tune для оптимизации:

```
from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler

def train_cnn(config):
    model = build_cnn(
        n_conv_layers=config['n_conv_layers'],
        n_filters=config['n_filters'],
        kernel_size=config['kernel_size']
    )
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    for epoch in range(10):
        train_model(model, optimizer, train_loader)
        val_loss = evaluate_model(model, val_loader)
        tune.report(loss=val_loss)

config = {
    'n_conv_layers': tune.choice([1, 2, 3]),
    'n_filters': tune.choice([16, 32, 64, 128]),
    'kernel_size': tune.choice([3, 5]),
    'lr': tune.loguniform(1e-4, 1e-2)
}

scheduler = ASHAScheduler(metric='loss', mode='min')
reporter = CLIReporter(metric_columns=['loss', 'training_iteration'])
analysis = tune.run(train_cnn, config=config, num_samples=20, scheduler=scheduler, progress_reporter=reporter)
```

Проанализируйте лучшие конфигурации и обучите финальную модель.
Дополнительно:

Добавьте Batch Normalization и оптимизируйте его использование.

https://www.kaggle.com/datasets/shaunthesheep/cifar10


# 3. Оптимизация гиперпараметров для больших языковых моделей (LLM)

## Задание 6: Оптимизация fine-tuning трансформера с Optuna

Описание:
Вы хотите оптимизировать процесс fine-tuning модели BERT для задачи классификации текстов.
Требования:

Используйте датасет SST-2 (Stanford Sentiment Treebank).
Определите пространство гиперпараметров:

learning_rate: [1e-5, 5e-5]
batch_size: [8, 32]
weight_decay: [0.0, 0.1]
warmup_steps: [0, 500]

Используйте Optuna для оптимизации:

```
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from datasets import load_dataset

def objective(trial):
    lr = trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.1)
    warmup_steps = trial.suggest_int('warmup_steps', 0, 500)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    dataset = load_dataset('glue', 'sst2')

    training_args = TrainingArguments(
        output_dir='./results',
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        num_train_epochs=3,
        evaluation_strategy='epoch'
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=dataset['train'], eval_dataset=dataset['validation'])
    trainer.train()
    eval_result = trainer.evaluate()
    return eval_result['eval_loss']

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)
```

Визуализируйте результаты и обучите финальную модель с лучшими параметрами.
Дополнительно:

Добавьте оптимизацию dropout для слоёв модели.

https://huggingface.co/datasets/glue


## Задание 7: Оптимизация генерации текста с LLM

Описание:
Вы хотите оптимизировать параметры генерации текста для модели GPT-2.
Требования:

Используйте датасет WritingPrompts.
Определите пространство гиперпараметров:

temperature: [0.1, 1.5]
top_k: [10, 100]
top_p: [0.1, 1.0]
repetition_penalty: [0.5, 2.0]

Используйте Optuna для оптимизации метрики качества генерации (например, perplexity):

```
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def objective(trial):
    temperature = trial.suggest_float('temperature', 0.1, 1.5)
    top_k = trial.suggest_int('top_k', 10, 100)
    top_p = trial.suggest_float('top_p', 0.1, 1.0)
    repetition_penalty = trial.suggest_float('repetition_penalty', 0.5, 2.0)

    model = GPT2LMHeadModel.from_pretrained('gpt2')
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

    inputs = tokenizer(prompt_text, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        max_new_tokens=50
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    perplexity = calculate_perplexity(model, tokenizer, generated_text)
    return perplexity

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
```

Проанализируйте лучшие параметры генерации.
Дополнительно:

Добавьте оценку качества генерации с помощью BLEU или ROUGE.

https://huggingface.co/datasets/writing_prompts


# 4. Продвинутые техники оптимизации

## Задание 8: Мультиобъективная оптимизация с Optuna

Описание:
Вы хотите оптимизировать модель по нескольким метрикам одновременно (например, точность и скорость предсказания).
Требования:

Используйте датасет Credit Card Fraud Detection.
Определите две метрики для оптимизации:

accuracy
inference_time

Используйте Optuna для мультиобъективной оптимизации:

```
from optuna.samplers import NSGAIISampler

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    accuracy = model.score(X_val, y_val)

    start_time = time.time()
    model.predict(X_val)
    inference_time = time.time() - start_time

    return accuracy, inference_time

study = optuna.create_study(directions=['maximize', 'minimize'], sampler=NSGAIISampler())
study.optimize(objective, n_trials=50)
```

Визуализируйте фронты Парето:

```
optuna.visualization.plot_pareto_front(study)
```

Дополнительно:

Добавьте третью метрику (например, model_size).

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud


## Задание 9: Оптимизация с учетом стоимости вычислений

Описание:
Вы хотите оптимизировать гиперпараметры с учётом стоимости вычислений (например, времени обучения).
Требования:

Используйте датасет NYC Taxi Trip Duration.
Определите пространство гиперпараметров для XGBoost:

n_estimators: [50, 500]
max_depth: [3, 10]
learning_rate: [0.01, 0.3]

Используйте Optuna с учётом времени обучения:

```
def objective(trial):
    start_time = time.time()
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3)
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    score = model.score(X_val, y_val)
    return score, training_time

study = optuna.create_study(directions=['maximize', 'minimize'])
study.optimize(objective, n_trials=50)
```

Проанализируйте компромисс между качеством и временем обучения.
Дополнительно:

Используйте Hyperband или BOHB для ускорения оптимизации.

https://www.kaggle.com/c/nyc-taxi-trip-duration/data


## Задание 10: Оптимизация ансамбля моделей

Описание:
Вы хотите оптимизировать веса и гиперпараметры ансамбля из нескольких моделей.
Требования:

Используйте датасет House Prices.
Постройте ансамбль из трёх моделей:

RandomForestRegressor
XGBRegressor
LinearRegression

Определите пространство гиперпараметров для каждой модели и веса ансамбля:

```
def objective(trial):
    # Гиперпараметры RandomForest
    rf_params = {
        'n_estimators': trial.suggest_int('rf_n_estimators', 50, 500),
        'max_depth': trial.suggest_int('rf_max_depth', 3, 20)
    }
    # Гиперпараметры XGBoost
    xgb_params = {
        'n_estimators': trial.suggest_int('xgb_n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('xgb_learning_rate', 0.01, 0.3)
    }
    # Веса ансамбля
    weights = [
        trial.suggest_float('weight_rf', 0, 1),
        trial.suggest_float('weight_xgb', 0, 1),
        trial.suggest_float('weight_lr', 0, 1)
    ]
    weights = [w / sum(weights) for w in weights]

    # Обучение моделей
    rf = RandomForestRegressor(**rf_params).fit(X_train, y_train)
    xgb = XGBRegressor(**xgb_params).fit(X_train, y_train)
    lr = LinearRegression().fit(X_train, y_train)

    # Предсказания ансамбля
    y_pred = weights[0] * rf.predict(X_val) + weights[1] * xgb.predict(X_val) + weights[2] * lr.predict(X_val)
    score = mean_squared_error(y_val, y_pred, squared=False)
    return score

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)
```

Визуализируйте важность гиперпараметров и весов моделей.
Дополнительно:

Добавьте в ансамбль нейронную сеть и оптимизируйте её архитектуру.

https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data


добавить задания по другим аспектам (например, оптимизация для временных рядов, байесовская оптимизация, оптимизация архитектуры трансформеров)